## airsim_dataset.py

In [ ]:
import os
import json
from PIL import Image
from scipy.ndimage import gaussian_filter

import cv2
import numpy as np
import torch

from vision_process import process_vision_info
from orthography import Orthophoto
from dataset.vis_data import visualize_waypoints


In [ ]:
video_frame_num = 5
target_interval = 30
data_list_json_paths = ["../data/train_data_sample.json"]
visualize = False
sigma = 20
visual_prompt = True

data_list = []
for file in data_list_json_paths:
    with open(file, "r") as f:
        datum = json.load(f)
    data_list += datum

data = data_list

ortho_processor = Orthophoto(granularity=0.3)

idx = 5

In [ ]:
def preprocess(image, pad_color=(0, 0, 0)):
    img_size = 784
    h, w = image.shape[:2]
    scale = img_size * 1.0 / max(h, w)
    new_h, new_w = h * scale, w * scale
    new_w = int(new_w + 0.5)
    new_h = int(new_h + 0.5)
    resized_image = cv2.resize(image, (new_w, new_h), interpolation=cv2.INTER_NEAREST)
    resized_hw = (new_h, new_w)

    pad_h = img_size - new_h
    pad_w = img_size - new_w
    padded_image = cv2.copyMakeBorder(resized_image, 0, pad_h, 0, pad_w,
                                    cv2.BORDER_CONSTANT, value=pad_color)
    return padded_image, resized_hw

def generate_prob_message_v2(pil_image, description):
    visual_prompt = True
    if isinstance(pil_image, np.ndarray):
        pil_image = Image.fromarray(pil_image)
    if isinstance(description, list):
        description = description[0]
    text_parts = description.split("The description of the target and its surrounding is shown below.")
    direction = text_parts[0].strip().split("Compass north corresponds to the top of the bird's-eye-view image.")[-1]
    direction = direction.strip()
    object_description = text_parts[-1].strip()

    prob_message = [{
        'role':'user',
        'content':[
            {'type':'image', 'image':pil_image},
            {'type':'text', 'text': "Task: Predict the probability distribution of the drone's future flight locations to search for the target."
            "Input Image: The image is an orthophoto map generated from the drone's past flight trajectory."},
            {'type':'text', 'text': "The green dots indicate past drone positions." if visual_prompt else " "},
            {'type':'text', 'text': "The top of the image corresponds to the north in the world coordinate system.\n" 
            f"Target Information: {direction}"
            f"The description of the target and its surrounding is shown below: {object_description}\n"
            "Objective: "
            "Analyze the provided orthophoto map and target information."
            "Predict the next flight location for the drone that maximize the probability of finding the target."
            "Output a probability map, indicating the likelihood of different regions in the orthophoto map being the optimal next flight destinations."
            }
        ]
    },
    {
        'role':'assistant',
        'content':[
            {'type':'image', 'image':pil_image}
        ]
    }
    ]
    return prob_message
    
def gaussian(target, img_size):
    sigma = 20
    h, w = img_size
    prob_map = np.zeros((h, w), dtype=np.float32)
    i, j = target
    i, j = min(round(i),h-1), min(round(j),w-1)
    prob_map[i, j] = 1
    i_1 = min(h-1, i+1)
    j_1 = min(w-1, j+1)
    i_2 = max(0, i-1)
    j_2 = max(0, j-1)
    prob_map[i, j_1] = 1
    prob_map[i, j_2] = 1
    prob_map[i_1, j] = 1
    prob_map[i_2, j] = 1

    sigma = max(h,w) // 25 if sigma is None else sigma
    prob_map = gaussian_filter(prob_map, sigma=sigma)
    return prob_map

def get_prob_map(ortho, coor_map, end, ortho_depth=None, delta_height=None):
    h, w = ortho.shape[:2]
    if ortho_depth is not None:
        depth_mask = (ortho_depth > delta_height).reshape(h,w)
    i, j = ortho_processor.world_to_pixel(end, coor_map=coor_map)
    i, j = min(max(round(i), 0), h-1), min(max(round(j), 0), w-1)
    prob_map_0 = np.zeros((h, w))
    prob_map_0[i, j] = 1

    yy, xx = np.ogrid[:h, :w]
    distances = np.sqrt((yy - i)**2 + (xx - j)**2)
    max_dist = np.sqrt(2*(h-1)**2)
    prob_map_0 = 1 - distances / max_dist
    prob_map_0[~depth_mask] = 0

    new_i, new_j = np.unravel_index(np.argmax(prob_map_0), (h, w))
    prob_map = gaussian((new_i, new_j), (h, w))
    # prob_map[~depth_mask] = 0
    prob_map = prob_map / (np.max(prob_map) + 1e-6)

    return prob_map, depth_mask

In [ ]:
data_info = data[idx]
traj_dir = data_info["traj_folder_path"]
depth_dir = os.path.join(traj_dir, "bevcamera_depth")
image_dir = os.path.join(traj_dir, "bevcamera")
log_dir = os.path.join(traj_dir, "log")
image_path = data_info["image_path"]

high_uav_pos_now = data_info["high_uav_pos_now"]
end_pos = data_info["end_pos"]
int_time = data_info["int_time"]
target_time = int_time+target_interval

description_path = os.path.join(traj_dir, "object_description_with_help.json")
with open(description_path, 'r') as f:
    description = json.load(f)
description = description[0]

image_files = sorted([f for f in os.listdir(image_dir)])
image_numbers = sorted([int(f.split('.')[0]) for f in image_files])
available_images = [t for t in image_numbers if t <= int_time]
available_num = len(available_images)
if available_num > video_frame_num:
    indices = [round(i * (available_num - 1) / (video_frame_num - 1)) for i in range(video_frame_num)]
    available_images = [available_images[i] for i in indices]
names = [f"{t:06d}" for t in available_images]

# historial orthography
frame_paths = [os.path.join(image_dir, f"{idx}.png") for idx in names]
log_paths = [os.path.join(log_dir, f"{idx}.json") for idx in names]
depth_paths = [os.path.join(depth_dir, f"{idx}.png") for idx in names]
positions = np.array([
            json.load(open(log_path, "r"))["sensors"]["state"]["position"] for log_path in log_paths
            ])
frames = np.array([cv2.imread(frame_path) for frame_path in frame_paths])
depths = np.array([cv2.imread(depth_path, cv2.IMREAD_UNCHANGED) for depth_path in depth_paths])

coord_3d_clouds = ortho_processor.project_images_to_3d(depths, positions)
merged_ortho, coor_map, merged_depth = ortho_processor.orthorectify(frames, coord_3d_clouds, depths)
merged_ortho = cv2.cvtColor(merged_ortho, cv2.COLOR_BGR2RGB)

In [ ]:
with open(os.path.join(traj_dir, "gt_waypoints.json"), "r") as f:
    gt_waypoints = json.load(f)
if len(gt_waypoints) > len(image_numbers):
    indices = [round(i * (len(gt_waypoints) - 1) / (len(image_numbers) - 1)) for i in range(len(image_numbers))]
else:
    indices = [i for i in range(len(gt_waypoints))]
index = image_numbers.index(int_time)
time_now = indices[index]
time_target = time_now + target_interval
waypoint_now = gt_waypoints[time_now]
if time_target < len(gt_waypoints):
    waypoint_target = gt_waypoints[time_target]
else:
    waypoint_target = end_pos
if visual_prompt:
    time_indexs = indices[:index]
    vis_waypoints = [gt_waypoints[n] for n in time_indexs]
    merged_ortho_prob = visualize_waypoints(vis_waypoints, coor_map, merged_ortho)
else:
    merged_ortho_prob = merged_ortho
prob_map, depth_mask = get_prob_map(merged_ortho, coor_map, waypoint_target, 
                                    ortho_depth=merged_depth, delta_height=waypoint_now[2]-high_uav_pos_now[2])
ortho_resize_pad, resized_hw = preprocess(merged_ortho_prob)
prob_map, _ = preprocess(prob_map)
depth_mask = depth_mask.astype(np.uint8)
depth_mask, _ = preprocess(depth_mask)
prob_message = generate_prob_message_v2(ortho_resize_pad, description)

prob_map = torch.from_numpy(prob_map)
prob_map = prob_map.unsqueeze(0)

In [ ]:
batch = {
            "prob_message": prob_message,
            "prob_map": prob_map.float(),
            "target_time": target_time,
            "traj_dir": traj_dir,
        }

In [ ]:
batch['traj_dir']

## Collate_fn

In [ ]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch
from torch.utils.data import DataLoader
from functools import partial


cache_path = "./weights/huggingface"
model_path = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
processor = AutoProcessor.from_pretrained(model_path, cache_dir=cache_path)

attn_impl = "sdpa"
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        attn_impl = "flash_attention_2"

model = AutoModelForImageTextToText.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    _attn_implementation=attn_impl,
    cache_dir=cache_path
).to("cuda")

In [ ]:
masks_list = []
messages_list = []
traj_folders = []
target_times = []

for data in [batch]:
    masks_list.append(data["prob_map"])
    message = data["prob_message"]
    messages_list.append(message)
    traj_folders.append(data["traj_dir"])
    target_times.append(data["target_time"])

texts = [processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in messages_list]
image_inputs, video_inputs = process_vision_info(messages_list)

inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )

In [ ]:
list(inputs.keys())

In [ ]:
inputs['pixel_values'].shape

In [ ]:
image_inputs

In [ ]:
image_inputs[0]

In [ ]:
image_inputs[1]

In [ ]:
input_ids_lists = inputs['input_ids'].tolist()
len(input_ids_lists[0])

In [ ]:
messages_list

In [ ]:
texts

In [ ]:
processor.tokenizer.encode("\nAssistant:")

In [ ]:
processor.tokenizer.encode("<end_of_utterance>\n")

In [ ]:
labels_ids = [-100] * len(input_ids_lists[0])
labels_ids

In [ ]:
processor

## Train pilot_llm

In [1]:
import argparse
import math
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'true'
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
# os.environ["FLASH_ATTN_DISABLE"] = "1"
from functools import partial
import time
import shutil

import datasets
import torch
from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import set_seed
from functools import partial
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from peft import get_peft_model, LoraConfig, PeftModel

from torch.utils.tensorboard import SummaryWriter

import transformers
from transformers import SchedulerType, get_scheduler
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.utils import check_min_version, send_example_telemetry
from transformers.utils.versions import require_version
from models_smolvlm.pilot_llm_smolvlm import PilotLLMSmolVLM
from dataset_smolvlm.airsim_dataset import AirSimDataset, collate_fn


/storage/project/r-cj124-0/sibidapo3/anxcnda/aeroduo/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
torch.cuda.get_device_capability()

(7, 5)

In [3]:
check_min_version("4.53.0")

logger = get_logger(__name__)

require_version("datasets>=2.0.0", "To fix: pip install -r examples/pytorch/semantic-segmentation/requirements.txt")

In [4]:
def parse_args():
    parser = argparse.ArgumentParser(description="Finetune a transformers model on a image semantic segmentation task")
    parser.add_argument(
        "--model_name_or_path",
        type=str,
        help="Path to a pretrained model or model identifier from huggingface.co/models.",
        default="HuggingFaceTB/SmolVLM2-2.2B-Instruct"
    )
    parser.add_argument(
        "--cache_path",
        type=str,
        help="default saved location of model if downloading",
        default="./weights/huggingface"
    )
    parser.add_argument(
        "--dataset_name",
        type=str,
        help="Name of the dataset on the hub",
        default=["../data/train_data_sample.json"],
    )
    parser.add_argument(
        "--reload_model_path",
        help="Name of the dataset on the hub.",
        default=None
    )
    parser.add_argument(
        "--num_token",
        type=int,
        default=2048,
        help="Maximum number of assistant image tokens used by the SmolVLM mask-query bank.",
    )
    parser.add_argument(
        "--mission",
        default="prob"
    )
    parser.add_argument(
        "--output_dir", 
        type=str, 
        help="Where to store the final model.",
        default="model_output"
    )
    parser.add_argument(
        "--lora",
        type=bool,
        help="if use lora to train",
        default=True,
    )
    parser.add_argument(
        "--lora_target_modules",
        type=list,
        default=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj']
    )
    parser.add_argument(
        "--per_device_train_batch_size",
        type=int,
        default=5,
        help="Batch size (per device) for the training dataloader.",
    )
    parser.add_argument(
        "--mixed_precision",
        type=str,
        default="bf16",
        help="",
    )
    parser.add_argument(
        "--learning_rate",
        type=float,
        default=5e-5,
        help="Initial learning rate (after the potential warmup period) to use.",
    )
    parser.add_argument(
        "--adam_beta1",
        type=float,
        default=0.9,
        help="Beta1 for AdamW optimizer",
    )
    parser.add_argument(
        "--adam_beta2",
        type=float,
        default=0.999,
        help="Beta2 for AdamW optimizer",
    )
    parser.add_argument(
        "--adam_epsilon",
        type=float,
        default=1e-8,
        help="Epsilon for AdamW optimizer",
    )
    parser.add_argument(
        "--num_train_epochs",
        type=int,
        default=3,
        help="Total number of training epochs to perform."
    )
    parser.add_argument(
        "--max_train_steps",
        type=int,
        default=10000,
        help="Total number of training steps to perform. If provided, overrides num_train_epochs.",
    )
    parser.add_argument( #########TODO#############
        "--gradient_accumulation_steps",
        type=int,
        default=1,
        help="Number of updates steps to accumulate before performing a backward/update pass.",
    )
    parser.add_argument(
        "--lr_scheduler_type",
        type=SchedulerType,
        default="cosine",
        help="The scheduler type to use.",
        choices=["linear", "cosine", "cosine_with_restarts", "polynomial", "constant", "constant_with_warmup"],
    )
    parser.add_argument(
        "--num_warmup_steps", type=int, default=0, help="Number of steps for the warmup in the lr scheduler."
    )
    parser.add_argument("--seed", type=int, default=None, help="A seed for reproducible training.")
    parser.add_argument(
        "--checkpointing_steps",
        type=str,
        default=1000,
        help="Whether the various states should be saved at the end of every n steps, or 'epoch' for each epoch.",
    )
    parser.add_argument(
        "--checkpoints_total_limit",
        type=int,
        default=3,
        help=""
    )
    parser.add_argument(
        "--resume_from_checkpoint",
        type=str,
        default=None,
        help="If the training should continue from a checkpoint folder.",
    )
    args, _ = parser.parse_known_args()

    if args.output_dir is not None:
        os.makedirs(args.output_dir, exist_ok=True)

    return args


In [5]:
import sys
sys.argv = ["ipykernel_launcher.py"]  # strip Jupyter-injected args

In [6]:
# ## def main

# args = parse_args()

# send_example_telemetry("run_semantic_segmentation_no_trainer", args)

# accelerator = Accelerator(gradient_accumulation_steps=args.gradient_accumulation_steps,
#                           mixed_precision=args.mixed_precision)

# logger.info(accelerator.state, main_process_only=False)
# if accelerator.is_local_main_process:
#     datasets.utils.logging.set_verbosity_warning()
#     transformers.utils.logging.set_verbosity_info()
# else:
#     datasets.utils.logging.set_verbosity_error()
#     transformers.utils.logging.set_verbosity_error()

# # If passed along, set the training seed now.
# # We set device_specific to True as we want different data augmentation per device.
# if args.seed is not None:
#     set_seed(args.seed, device_specific=True)

# if accelerator.is_main_process:
#     if args.output_dir is not None:
#         os.makedirs(args.output_dir, exist_ok=True)
# accelerator.wait_for_everyone()


# current_time = time.strftime("%Y%m%d-%H%M%S", time.localtime())
# log_dir = os.path.join(args.output_dir, current_time)
# tb_writer = SummaryWriter(log_dir=log_dir)

# weight_dtype = torch.float32
# if accelerator.mixed_precision == "fp16":
#     weight_dtype = torch.float16
#     args.mixed_precision = accelerator.mixed_precision
# elif accelerator.mixed_precision == "bf16":
#     weight_dtype = torch.bfloat16
#     args.mixed_precision = accelerator.mixed_precision

# #from models.pilot_llm_smolvlm import PilotLLMSmolVLM

# processor = AutoProcessor.from_pretrained(args.model_name_or_path, cache_dir=args.cache_path, padding_side="right")
# attn_impl = "sdpa"
# if torch.cuda.is_available():
#     major, _ = torch.cuda.get_device_capability()
#     if major >= 8:
#         attn_impl = "flash_attention_2"
# # model = PilotLLMSmolVLM.from_pretrained(
# #     args.model_name_or_path,
# #     _attn_implementation=attn_impl,
# #     cache_dir=args.cache_path,
# #     num_image_token=args.num_token,
# #     )
# model = AutoModelForImageTextToText.from_pretrained(
#     args.model_name_or_path,
#     torch_dtype=torch.bfloat16,
#     _attn_implementation=attn_impl,
#     cache_dir=args.cache_path
# )

# # load pretrain model（seg and depth pretraining
# if args.reload_model_path is not None:
#     model = PeftModel.from_pretrained(model, args.reload_model_path)
#     weights = torch.load(os.path.join(args.reload_model_path, "pytorch_model/mp_rank_00_model_states.pt"))
#     model.load_state_dict(weights['module'], strict=False)

#     # If loading a trained main model, do not rewrite a set of lora
#     model = model.merge_and_unload()
#     if accelerator.is_main_process:
#         del weights

# if args.lora:
#     lora_config = LoraConfig(
#         r=8,
#         target_modules=args.lora_target_modules,
#         task_type="CAUSAL_LM",
#     )
#     model = get_peft_model(model, lora_config)

In [7]:
## def main

args = parse_args()

send_example_telemetry("run_semantic_segmentation_no_trainer", args)

accelerator = Accelerator(gradient_accumulation_steps=args.gradient_accumulation_steps,
                          mixed_precision=args.mixed_precision)

logger.info(accelerator.state, main_process_only=False)
if accelerator.is_local_main_process:
    datasets.utils.logging.set_verbosity_warning()
    transformers.utils.logging.set_verbosity_info()
else:
    datasets.utils.logging.set_verbosity_error()
    transformers.utils.logging.set_verbosity_error()

# If passed along, set the training seed now.
# We set device_specific to True as we want different data augmentation per device.
if args.seed is not None:
    set_seed(args.seed, device_specific=True)

if accelerator.is_main_process:
    if args.output_dir is not None:
        os.makedirs(args.output_dir, exist_ok=True)
accelerator.wait_for_everyone()


current_time = time.strftime("%Y%m%d-%H%M%S", time.localtime())
log_dir = os.path.join(args.output_dir, current_time)
tb_writer = SummaryWriter(log_dir=log_dir)

weight_dtype = torch.float32
if accelerator.mixed_precision == "fp16":
    weight_dtype = torch.float16
    args.mixed_precision = accelerator.mixed_precision
elif accelerator.mixed_precision == "bf16":
    weight_dtype = torch.bfloat16
    args.mixed_precision = accelerator.mixed_precision

#from models.pilot_llm_smolvlm import PilotLLMSmolVLM

processor = AutoProcessor.from_pretrained(args.model_name_or_path, cache_dir=args.cache_path, padding_side="right")
attn_impl = "sdpa"
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        attn_impl = "flash_attention_2"
# model = PilotLLMSmolVLM.from_pretrained(
#     args.model_name_or_path,
#     _attn_implementation=attn_impl,
#     cache_dir=args.cache_path,
#     num_image_token=args.num_token,
#     )
model = PilotLLMSmolVLM.from_pretrained(
        args.model_name_or_path,
        num_image_token=args.num_token,
    )


# load pretrain model（seg and depth pretraining
if args.reload_model_path is not None:
    model = PeftModel.from_pretrained(model, args.reload_model_path)
    weights = torch.load(os.path.join(args.reload_model_path, "pytorch_model/mp_rank_00_model_states.pt"))
    model.load_state_dict(weights['module'], strict=False)

    # If loading a trained main model, do not rewrite a set of lora
    model = model.merge_and_unload()
    if accelerator.is_main_process:
        del weights

if args.lora:
    lora_config = LoraConfig(
        r=8,
        target_modules=args.lora_target_modules,
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)

loading configuration file processor_config.json from cache at ./weights/huggingface/models--HuggingFaceTB--SmolVLM2-2.2B-Instruct/snapshots/482adb537c021c86670beed01cd58990d01e72e4/processor_config.json
loading configuration file preprocessor_config.json from cache at ./weights/huggingface/models--HuggingFaceTB--SmolVLM2-2.2B-Instruct/snapshots/482adb537c021c86670beed01cd58990d01e72e4/preprocessor_config.json
Image processor SmolVLMImageProcessor {
  "do_convert_rgb": true,
  "do_image_splitting": true,
  "do_normalize": true,
  "do_pad": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.5,
    0.5,
    0.5
  ],
  "image_processor_type": "SmolVLMImageProcessor",
  "image_std": [
    0.5,
    0.5,
    0.5
  ],
  "max_image_size": {
    "longest_edge": 384
  },
  "processor_class": "SmolVLMProcessor",
  "resample": 1,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "longest_edge": 1536
  },
  "video_sampling": {
    "fps": 1,
    "max_frames": 64,
    "vi

In [8]:
model.print_trainable_parameters()

trainable params: 10,536,960 || all params: 2,265,714,546 || trainable%: 0.4650612328284006


In [9]:
for n, p in model.named_parameters():
    if any(
        [
            x in n for x in ["depth_head"]
        ]
    ):
        print("n: ", n, "p.shape: ", p.shape)
        print(p.requires_grad)

n:  base_model.model.depth_head.mlp.0.weight p.shape:  torch.Size([1024, 2048])
False
n:  base_model.model.depth_head.mlp.0.bias p.shape:  torch.Size([1024])
False
n:  base_model.model.depth_head.mlp.2.weight p.shape:  torch.Size([1, 1024])
False
n:  base_model.model.depth_head.mlp.2.bias p.shape:  torch.Size([1])
False


In [10]:
args.output_dir

'model_output'

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(accelerator.device, weight_dtype)

In [12]:
train_dataset = AirSimDataset(args.dataset_name, 
                              video_frame_num=5,
                              target_interval=30)

train_dataloader = DataLoader(
    train_dataset,
    batch_size=2,#args.per_device_train_batch_size,
    collate_fn=partial(collate_fn, processor=processor, mission=args.mission)
)

dataset has 53137 samples load from file ['../data/train_data_sample.json']


In [13]:
model.train()
optimizer = torch.optim.AdamW(
        list(model.parameters()),
        lr=args.learning_rate,
        betas=[args.adam_beta1, args.adam_beta2],
        eps=args.adam_epsilon,
    )

In [14]:
if accelerator.is_main_process:
    rec_txt1 = open(os.path.join(args.output_dir, 'rec_para.txt'), 'w')
    rec_txt2 = open(os.path.join(args.output_dir,'rec_para_train.txt'), 'w')
    for name, para in model.named_parameters():
        if para.requires_grad is False:
            rec_txt1.write(f'{name}\n')
        else:
            rec_txt2.write(f'{name}\n')
    rec_txt1.close()
    rec_txt2.close()

In [15]:
checkpointing_steps = args.checkpointing_steps
checkpointing_steps = int(checkpointing_steps)
overrode_max_train_steps = False
num_update_steps_per_epoch = math.ceil(len(train_dataloader) / args.gradient_accumulation_steps)
if args.max_train_steps is None:
    args.max_train_steps = args.num_train_epochs * num_update_steps_per_epoch
    overrode_max_train_steps = True

In [16]:
lr_scheduler = get_scheduler(
        name=args.lr_scheduler_type,
        optimizer=optimizer,
        num_warmup_steps=args.num_warmup_steps * accelerator.num_processes,
        num_training_steps=args.max_train_steps
        if overrode_max_train_steps
        else args.max_train_steps * accelerator.num_processes,
    )

In [17]:
model, optimizer, train_dataloader, lr_scheduler = accelerator.prepare(
        model, optimizer, train_dataloader, lr_scheduler
    )
# We need to recalculate our total training steps as the size of the training dataloader may have changed.
num_update_steps_per_epoch = math.ceil(len(train_dataloader) / args.gradient_accumulation_steps)
if overrode_max_train_steps:
    args.max_train_steps = args.num_train_epochs * num_update_steps_per_epoch
# Afterwards we recalculate our number of training epochs
args.num_train_epochs = math.ceil(args.max_train_steps / num_update_steps_per_epoch)

# Train!
total_batch_size = args.per_device_train_batch_size * accelerator.num_processes * args.gradient_accumulation_steps

In [18]:
accelerator.sync_gradients

True

In [ ]:
for step, batch in enumerate(train_dataloader):
    inputs, kwargs = batch
    if step >= 0:
        break

inputs = inputs.to(device, weight_dtype)

[Errno 2] No such file or directory: 'data/Hal-13k/Carla_Town10HD/62a365ab-6b00-462c-b092-8b264bf319a8/object_description_with_help.json'
[Errno 2] No such file or directory: 'data/Hal-13k/Carla_Town10HD/62a365ab-6b00-462c-b092-8b264bf319a8/object_description_with_help.json'
[Errno 2] No such file or directory: 'data/Hal-13k/Carla_Town10HD/62a365ab-6b00-462c-b092-8b264bf319a8/object_description_with_help.json'
[Errno 2] No such file or directory: 'data/Hal-13k/Carla_Town10HD/62a365ab-6b00-462c-b092-8b264bf319a8/object_description_with_help.json'
[Errno 2] No such file or directory: 'data/Hal-13k/Carla_Town10HD/62a365ab-6b00-462c-b092-8b264bf319a8/object_description_with_help.json'
[Errno 2] No such file or directory: 'data/Hal-13k/Carla_Town10HD/62a365ab-6b00-462c-b092-8b264bf319a8/object_description_with_help.json'
[Errno 2] No such file or directory: 'data/Hal-13k/Carla_Town10HD/62a365ab-6b00-462c-b092-8b264bf319a8/object_description_with_help.json'
[Errno 2] No such file or director


KeyboardInterrupt



In [ ]:
list(inputs.keys())

['pixel_values', 'pixel_attention_mask', 'input_ids', 'attention_mask']

In [ ]:
list(kwargs.keys())

['masks_list',
 'labels',
 'mission',
 'image_inputs',
 'traj_folders',
 'pred_time']

In [ ]:
inputs['pixel_values'].shape

torch.Size([2, 34, 3, 384, 384])

In [ ]:
inputs['attention_mask']

tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]], device='cuda:0')

## pilot_llm_smolvlm

In [ ]:
model.config

vision_config is None, using default vision config
text_config is None, using default text config


SmolVLMConfig {
  "architectures": [
    "SmolVLMForConditionalGeneration"
  ],
  "image_token_id": 49190,
  "model_type": "smolvlm",
  "pad_token_id": 128002,
  "scale_factor": 3,
  "text_config": {
    "_flash_attn_2_enabled": true,
    "_name_or_path": "None",
    "architectures": [
      "VLlama3ForCausalLM"
    ],
    "attention_bias": false,
    "attention_dropout": 0.0,
    "head_dim": 64,
    "hidden_act": "silu",
    "hidden_size": 2048,
    "initializer_range": 0.02,
    "intermediate_size": 8192,
    "max_position_embeddings": 8192,
    "mlp_bias": false,
    "model_type": "llama",
    "neftune_noise_alpha": 0.0,
    "num_attention_heads": 32,
    "num_hidden_layers": 24,
    "num_key_value_heads": 32,
    "pad_token_id": 2,
    "perceiver_config": {
      "_name_or_path": "",
      "add_cross_attention": false,
      "architectures": null,
      "attention_dropout": 0.0,
      "bad_words_ids": null,
      "begin_suppress_tokens": null,
      "bos_token_id": null,
      "chu

In [ ]:
import torch.nn as nn
mask_query = nn.Parameter(
            torch.zeros([2048, model.model.config.text_config.hidden_size])).to('cuda')

In [ ]:
input_ids = inputs['input_ids']
attention_mask = inputs['attention_mask']
position_ids = None
past_key_values = None
inputs_embeds = None
pixel_values = inputs['pixel_values']
pixel_attention_mask = inputs['pixel_attention_mask']
image_hidden_states = None
labels = kwargs['labels']
masks_list = kwargs['masks_list']
use_cache = None
output_attentions = None 
output_hidden_states = None
cache_position = None
return_dict = None
logits_to_keep = None
is_inference = None

In [ ]:
output_attentions = output_attentions if output_attentions is not None else model.config.output_attentions
output_hidden_states = (
    output_hidden_states if output_hidden_states is not None else model.config.output_hidden_states
)
return_dict = return_dict if return_dict is not None else model.config.use_return_dict

bs = input_ids.shape[0]
labels = labels.to(input_ids.device)

# special_mask = (labels == model.config.image_token_id) & (input_ids == model.config.image_token_id)
# mask_counts = special_mask.sum(dim=1)
torch.cuda.empty_cache()
if inputs_embeds is None:
    inputs_embeds = model.model.model.text_model.get_input_embeddings()(input_ids)
    if pixel_values is not None:
        with torch.no_grad():
            image_hidden_states = model.model.model.get_image_features(pixel_values, pixel_attention_mask)
        n_image_tokens = (input_ids == model.model.config.image_token_id).sum().item()
        n_image_features = image_hidden_states.shape[0]*image_hidden_states.shape[1]

        if n_image_tokens != n_image_features:
            raise ValueError(
                f"Image features and image tokens do not match: tokens: {n_image_tokens}, features {n_image_features}"
            )
        
        image_hidden_states = image_hidden_states.to(inputs_embeds.device, inputs_embeds.dtype)
        inputs_embeds = model.model.model.inputs_merger(
            input_ids = input_ids,
            inputs_embeds = inputs_embeds,
            image_hidden_states = image_hidden_states
        )

In [ ]:
special_mask = (input_ids == model.model.config.image_token_id) & (labels == model.model.config.image_token_id)
mask_counts = special_mask.sum(dim=1)
max_needed = int(mask_counts.max().item()) if mask_counts.numel() > 0 else 0
if max_needed > mask_query.shape[0]:
    raise ValueError(
        f"Need {max_needed} mask query tokens, but the query bank only has {mask_query.shape[0]}. "
        "Increase `num_image_token` when loading the model."
    )
for i in range(bs):
    count = int(mask_counts[i].item())
    if count == 0:
        continue
    mask_indices = special_mask[i].nonzero(as_tuple=True)[0]
    inputs_embeds[i, mask_indices] = mask_query[:count].to(inputs_embeds.dtype)

if attention_mask is not None:
    attention_mask = attention_mask.to(inputs_embeds.device)


In [21]:
outputs = model.model.model.text_model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            position_ids=position_ids,
            past_key_values=past_key_values,
            use_cache=use_cache,
            output_attentions=output_attentions,
            output_hidden_states=output_hidden_states,
            return_dict=return_dict,
            cache_position=cache_position,
            **kwargs,
        )

In [22]:
int(
    ((model.model.config.vision_config.image_size // model.model.config.vision_config.patch_size) ** 2) / (model.model.config.scale_factor**2)
)

81

In [23]:
hidden_states = outputs.last_hidden_state

mask_query = hidden_states[special_mask].view(bs, mask_counts[0], model.model.config.text_config.hidden_size)

In [24]:
mask_indices.shape

torch.Size([1377])

## Mask Head Params

In [25]:
masks_list[0].shape[-2:] 

torch.Size([784, 784])

In [27]:
hidden_size = model.model.config.text_config.hidden_size
image_seq_len = int(((model.model.config.vision_config.image_size // model.model.config.vision_config.patch_size) ** 2) / (model.model.config.scale_factor**2))
target_hw = masks_list[0].shape[-2:]
for i in range(bs):
    mask_indices = special_mask[i].nonzero(as_tuple=True)[0]
    mask_head_in = hidden_states[i, mask_indices]

mlp = nn.Sequential(
        nn.Linear(hidden_size, hidden_size // 2),
        nn.ReLU(),
        nn.Linear(hidden_size // 2, 1),
    ).to(device).type(mask_head_in.dtype)

In [28]:
## _tokens_to_blocks

num_tokens = mask_head_in.shape[0]
if num_tokens % image_seq_len != 0:
    raise ValueError(
        f"Expected mask tokens to be divisible by image_seq_len, got {num_tokens} and {image_seq_len}."
    )
block_side = math.isqrt(image_seq_len)
if block_side * block_side != image_seq_len:
    raise ValueError(f"Expected a square image_seq_len, got {image_seq_len}.")

num_blocks = num_tokens // image_seq_len
token_logits = mlp(mask_head_in).squeeze(-1)
token_logits = token_logits.view(num_blocks, block_side, block_side)

In [ ]:
## _stitch_blocks
import torch.nn.functional as F
block_maps = token_logits
num_blocks, block_h, block_w = block_maps.shape
# if num_blocks == 1:
# return block_maps[0]
local_blocks = num_blocks - 1
local_side = math.isqrt(local_blocks)
if local_side * local_side != local_blocks:
    raise ValueError(
        "SmolVLM split-image decoding expects `rows * cols + 1` blocks "
        f"(local crops + global crop), got {num_blocks}."
    )
local_map = block_maps[:-1].view(local_side, local_side, block_h, block_w)
local_map = local_map.permute(0, 2, 1, 3).reshape(local_side * block_h, local_side * block_w)

global_map = block_maps[-1][None, None]
global_map = F.interpolate(
            global_map,
            size=local_map.shape,
            mode="bilinear",
            align_corners=False,
        )[0, 0]
output = 0.5 * (local_map + global_map)

In [55]:
##forward
coarse_mask = output
pred_mask = F.interpolate(
    coarse_mask[None, None],
    size=target_hw,
    mode = "bilinear",
    align_corners=False
)


In [58]:
pred_mask[0].shape

torch.Size([1, 784, 784])

In [54]:
coarse_mask[None,None] == coarse_mask

tensor([[[[True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          ...,
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True],
          [True, True, True,  ..., True, True, True]]]], device='cuda:0')

In [40]:
block_maps[:-1].shape

torch.Size([16, 9, 9])

In [40]:
query.shape

torch.Size([2, 1377, 2048])

In [66]:
model.model.config.text_config.hidden_size

2048

In [67]:
mask_counts

tensor([1377, 1377], device='cuda:0')

In [ ]:
hidden_states[special_mask].shape

torch.Size([2754, 2048])

In [62]:
special_mask.shape

torch.Size([2, 3043])

In [42]:
hidden_states[input_ids == model.model.config.image_token_id]

tensor([[ 1.2266, -0.4922, -0.9023,  ...,  1.1094,  0.3008,  0.3574],
        [ 1.3281, -0.0737, -0.9023,  ...,  2.4531,  0.1592,  0.2158],
        [ 1.5469,  0.5625, -0.9453,  ...,  1.4922, -0.0645, -0.7578],
        ...,
        [ 1.5391,  0.0981, -1.8984,  ..., -0.0757, -0.1914,  1.1953],
        [ 1.5781,  0.0879, -1.8516,  ..., -0.0342, -0.1787,  1.1484],
        [ 1.6406,  0.0713, -1.8359,  ..., -0.0688, -0.1680,  1.1328]],
       device='cuda:0', dtype=torch.bfloat16, grad_fn=<IndexBackward0>)

In [41]:
hidden_states[labels == model.model.config.image_token_id].shape

torch.Size([2754, 2048])

In [43]:
n_image_tokens = (input_ids == model.model.config.image_token_id).sum().item()

In [45]:
(labels == model.model.config.image_token_id).sum().item()

2754

In [64]:
input_ids.shape

torch.Size([2, 3043])

In [46]:
5508-2754

2754

In [50]:
((input_ids == model.model.config.image_token_id) & (labels == model.model.config.image_token_id)).sum().item()

2754

In [51]:
special_mask = (input_ids == model.model.config.image_token_id) & (labels == model.model.config.image_token_id)


In [53]:
special_mask.sum()

tensor(2754, device='cuda:0')

In [ ]:
image_mask = (input_ids == model.model.config.image_token_id).unsqueeze(-1).expand_as(inputs_embeds)

In [ ]:
inputs_embeds.shape

In [ ]:
image_hidden_states.shape

In [ ]:
attention_mask

## SmolVLM PilotLLM path

`collate_fn` returns `(inputs, kwargs)`.
`inputs` is the SmolVLM backbone input and can be forwarded as-is: `input_ids`, `attention_mask`, `pixel_values`, and `pixel_attention_mask`.
`kwargs` carries the task supervision used by the custom PilotLLM head: `labels` identifies the assistant image-token slots, and `masks_list` is the probability-map target.


In [ ]:
from models.pilot_llm_smolvlm import PilotLLMSmolVLM

`PilotLLMSmolVLM` mirrors `models/pilot_llm.py` for the Qwen model, but adapts the image-token handling to SmolVLM.
SmolVLM2 splits a large image into local crops plus one global crop, so the current `784x784` orthophoto becomes `17` image blocks, each with `81` `<image>` tokens.
That is why the assistant-side supervision here lands on `1377 = 17 x 81` image-token positions instead of the fixed `784` Qwen layout.


In [ ]:
base_model = model.get_base_model() if hasattr(model, "get_base_model") else model
labels = kwargs["labels"].to(inputs["input_ids"].device)
image_token_id = base_model.config.image_token_id if hasattr(base_model, "config") else base_model.model.config.image_token_id
image_seq_len = base_model.image_seq_len if hasattr(base_model, "image_seq_len") else base_model.model.image_seq_len
special_mask = (inputs["input_ids"] == image_token_id) & (labels == image_token_id)

In [ ]:
special_mask.sum(dim=1), image_seq_len

In [ ]:
model_kwargs = {
    "labels": kwargs["labels"].to(device),
    "masks_list": [mask.to(device) for mask in kwargs["masks_list"]],
    "mission": kwargs["mission"],
}

# This is the SmolVLM equivalent of the Qwen PilotLLM training call.
pred_masks, loss = model(**inputs, **model_kwargs)

In [ ]:
kwargs["masks_list"][0].shape

In [ ]:
pred_masks.shape

In [ ]:
loss

In [ ]:
# `image_inputs`, `traj_folders`, and `pred_time` stay outside the forward call.
# They are still available in `kwargs` for logging, visualization, or evaluation.

In [ ]:
from transformers.models.smolvlm.modeling_smolvlm import SmolVLMForConditionalGeneration